<a href="https://colab.research.google.com/github/abdelwhabkamal/EPL-Matches-Prediction/blob/main/EPL-Matches-Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# English Premier League (EPL) Match Outcome Prediction

This notebook demonstrates an end-to-end machine learning workflow to predict English Premier League match outcomes (specifically, home wins) using historical match data and rolling team form statistics.

We focus on a **Binary Classification** task predicting whether the Home Team wins (Target = `1`) or fails to win (Draw or Away Win, Target = `0`).
```

```markdown
## 1. Imports and Data Loading

We start by importing our essential data manipulation, evaluation, and machine learning packages.
```

In [339]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

```markdown
### Loading the EPL Dataset

We load the match database and filter records starting from the **22/23 Season** to construct robust chronological momentum features.
```

In [340]:
matches = pd.read_csv('/content/EPL_Match_Data_2000_2026.csv')
matches_recent = matches[matches['Season'] >= 2223].copy()
print(f"Loaded {len(matches_recent)} records from Season 22/23 onwards.")
display(matches_recent.head())

Loaded 1520 records from Season 22/23 onwards.


,Season,Year,Month,Day,WeekDay,Time,Round,HomeTeam,AwayTeam,HomeGoal,...,EuropaLeagueHome,EuropaLeagueAway,ConferenceLeagueHome,ConferenceLeagueAway,SuperCupHome,SuperCupAway,ClubWorldCupHome,ClubWorldCupAway,IntercontinentalCupHome,IntercontinentalCupAway
8360,2223,2022,8,5,Sun,16:30,17.0,Crystal Palace,Arsenal,0,...,No,No,No,No,No,No,No,No,No,No
8361,2223,2022,8,6,Sun,16:30,17.0,Tottenham,Southampton,4,...,No,No,No,No,No,No,No,No,No,No
8362,2223,2022,8,6,Mon,20:00,17.0,Everton,Chelsea,0,...,No,No,No,No,No,No,No,No,No,No
8363,2223,2022,8,6,Mon,20:00,17.0,Newcastle,Nottingham,2,...,No,No,No,No,No,No,No,No,No,No
8364,2223,2022,8,6,Mon,20:00,17.0,Fulham,Liverpool,2,...,No,No,No,No,No,No,No,No,No,No


```markdown
## 2. Preprocessing & Temporal Structuring

To prevent data leakage and evaluate our models honestly, matches must be sorted strictly chronologically.
```

In [341]:
# Reset indices to cleanly map temporal alignments
matches_reset = matches_recent.reset_index(drop=True).reset_index().rename(columns={'index': 'MatchIdx'})

# Map weekdays, home/away categorical identifiers, and kick-off hour configurations
day_map = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
matches_reset['DayCode'] = matches_reset['WeekDay'].map(day_map)
matches_reset['HomeCode'] = matches_reset['HomeTeam'].astype('category').cat.codes
matches_reset['AwayCode'] = matches_reset['AwayTeam'].astype('category').cat.codes
matches_reset['Hour'] = matches_reset['Time'].str.replace(":", "").str.slice(0, 2).astype("int")

# Construct our binary target variable (1 = Home Win, 0 = Draw or Away Win)
matches_reset['TargetBinary'] = (matches_reset['Result'] == 'H').astype(int)
print("Categorical features and binary target mapped successfully.")

Categorical features and binary target mapped successfully.


```markdown
## 3. Dynamic Chronological Form (Rolling Stats)

We model teams' dynamic rolling momentum by computing a team-by-team chronology of performance metrics across all of their home and away matches combined.
```

In [342]:
def get_team_chronological_stats(df):
    df_sorted = df.sort_values(['Season', 'Month', 'Day', 'Time']).copy()
    records = []
    for idx, row in df_sorted.iterrows():
        records.append({
            'MatchIdx': idx,
            'Team': row['HomeTeam'],
            'Opponent': row['AwayTeam'],
            'IsHome': 1,
            'Season': row['Season'],
            'Month': row['Month'],
            'Day': row['Day'],
            'Time': row['Time'],
            'GF': row['HomeGoal'],
            'GA': row['AwayGoal'],
            'Shots': row['HomeShot'],
            'ShotsOnTarget': row['HomeOnTargetShot'],
            'Corners': row['HomeCorner'],
            'Fouls': row['HomeFoul'],
            'XG': row['XGHome'],
            'Result': 'W' if row['Result'] == 'H' else ('L' if row['Result'] == 'A' else 'D')
        })
        records.append({
            'MatchIdx': idx,
            'Team': row['AwayTeam'],
            'Opponent': row['HomeTeam'],
            'IsHome': 0,
            'Season': row['Season'],
            'Month': row['Month'],
            'Day': row['Day'],
            'Time': row['Time'],
            'GF': row['AwayGoal'],
            'GA': row['HomeGoal'],
            'Shots': row['AwayShot'],
            'ShotsOnTarget': row['AwayOnTargetShot'],
            'Corners': row['AwayCorner'],
            'Fouls': row['AwayFoul'],
            'XG': row['XGAway'],
            'Result': 'W' if row['Result'] == 'A' else ('L' if row['Result'] == 'H' else 'D')
        })
    return pd.DataFrame(records)

team_history = get_team_chronological_stats(matches_reset)
print(f"Extracted {len(team_history)} team-match history records.")

Extracted 3040 team-match history records.


```markdown
### Computing 3-Match Rolling Momentum
```

In [343]:
def compute_team_rolling(group):
    metrics = ['GF', 'GA', 'Shots', 'ShotsOnTarget', 'Corners', 'Fouls', 'XG']
    rolling_cols = [f'{m}_rolling' for m in metrics]
    group = group.sort_values(['Season', 'Month', 'Day', 'Time'])
    # Closed left avoids data leakage by excluding the current match
    group[rolling_cols] = group[metrics].rolling(3, closed='left').mean()
    return group

team_history_rolling = team_history.groupby('Team', group_keys=False).apply(compute_team_rolling)
print("Dynamic rolling metrics computed successfully.")

Dynamic rolling metrics computed successfully.


```markdown
### Aligning and Merging Rolling Features Back to Main Matches
```

In [344]:
metrics = ['GF', 'GA', 'Shots', 'ShotsOnTarget', 'Corners', 'Fouls', 'XG']
rolling_cols = [f'{m}_rolling' for m in metrics]

home_stats = team_history_rolling[team_history_rolling['IsHome'] == 1].copy()[['MatchIdx'] + rolling_cols].rename(columns={c: f'Home_{c}' for c in rolling_cols})
away_stats = team_history_rolling[team_history_rolling['IsHome'] == 0].copy()[['MatchIdx'] + rolling_cols].rename(columns={c: f'Away_{c}' for c in rolling_cols})

matches_combined_fixed = matches_reset.merge(home_stats, on='MatchIdx').merge(away_stats, on='MatchIdx')
all_rolling_cols = [f'Home_{c}' for c in rolling_cols] + [f'Away_{c}' for c in rolling_cols]
matches_combined_fixed = matches_combined_fixed.dropna(subset=all_rolling_cols)

print(f"Aligned Feature matrix size: {matches_combined_fixed.shape}")

Aligned Feature matrix size: (1472, 105)


```markdown
## 4. Injecting Advanced Strategic Domain Features

We add **Rest Days** (calculating schedule gaps between consecutive matches capped at 14 days) and **H2H Advantage** (historical Head-to-Head form) to enhance the predictive signal.
```

In [345]:
def add_advanced_domain_features(df_matches, df_history):
    df_history_sorted = df_history.copy()
    match_years = df_matches.set_index('MatchIdx')['Year'].to_dict()
    df_history_sorted['Year'] = df_history_sorted['MatchIdx'].map(match_years).fillna(2024).astype(int)
    df_history_sorted = df_history_sorted.sort_values(['Year', 'Month', 'Day', 'Time']).copy()
    df_history_sorted['Date'] = pd.to_datetime(df_history_sorted[['Year', 'Month', 'Day']])
    df_history_sorted = df_history_sorted.sort_values(['Team', 'Date'])

    df_history_sorted['LastMatchDate'] = df_history_sorted.groupby('Team')['Date'].shift(1)
    df_history_sorted['RestDays'] = (df_history_sorted['Date'] - df_history_sorted['LastMatchDate']).dt.days
    df_history_sorted['RestDays'] = df_history_sorted['RestDays'].apply(lambda x: 7.0 if pd.isna(x) or x < 0 else min(x, 14.0))

    home_rest = df_history_sorted[df_history_sorted['IsHome'] == 1][['MatchIdx', 'RestDays']].rename(columns={'RestDays': 'Home_RestDays'})
    away_rest = df_history_sorted[df_history_sorted['IsHome'] == 0][['MatchIdx', 'RestDays']].rename(columns={'RestDays': 'Away_RestDays'})

    df_merged = df_matches.merge(home_rest, on='MatchIdx', how='left')
    df_merged = df_merged.merge(away_rest, on='MatchIdx', how='left')

    df_merged['Home_RestDays'] = df_merged['Home_RestDays'].fillna(7.0)
    df_merged['Away_RestDays'] = df_merged['Away_RestDays'].fillna(7.0)
    df_merged['H2H_Advantage'] = df_merged['H2HForm'].apply(lambda val: 1.0 if 'W' in str(val).upper() else (-1.0 if 'L' in str(val).upper() else 0.0))
    return df_merged

matches_enhanced = add_advanced_domain_features(matches_combined_fixed, team_history_rolling)
print("Domain Features Integrated! Final dataset shape:", matches_enhanced.shape)

Domain Features Integrated! Final dataset shape: (1472, 108)


```markdown
## 5. Hyperparameter Tuning Analysis & GridSearchCV

In previous workflows, we conducted extensive searches to locate robust and stable estimators that handle chronological validation effectively. Below, we perform a target grid search over key hyperparameters to optimize our classifiers and maximize prediction accuracy.
```

In [346]:
# Prepare features and chronological split (Train 80%, Test 20% to avoid temporal leakage)
enhanced_predictors = ['HomeCode', 'AwayCode', 'DayCode', 'Hour'] + all_rolling_cols + ['Home_RestDays', 'Away_RestDays', 'H2H_Advantage']
data_sorted = matches_enhanced.sort_values(['Season', 'Month', 'Day'])
train, test = train_test_split(data_sorted, test_size=0.20, shuffle=False)

print(f"Training set: {train.shape[0]} matches | Testing set: {test.shape[0]} matches")

Training set: 1177 matches | Testing set: 295 matches


In [347]:
# Tuning Random Forest Classifier via GridSearchCV
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=1),
    param_grid={
        'n_estimators': [100, 150],
        'max_depth': [6, 8, 10],
        'min_samples_split': [10, 15]
    },
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)
rf_grid.fit(train[enhanced_predictors], train["TargetBinary"])
print("Best RF Parameters Found:", rf_grid.best_params_)
print(f"RF Tuned Grid Accuracy: {rf_grid.best_score_:.4f}")

Best RF Parameters Found: {'max_depth': 6, 'min_samples_split': 15, 'n_estimators': 150}
RF Tuned Grid Accuracy: 0.6449


In [348]:
# Tuning HistGradientBoosting Classifier via GridSearchCV
hgb_grid = GridSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=1),
    param_grid={
        'learning_rate': [0.01, 0.05],
        'max_iter': [80, 120],
        'max_depth': [4, 6],
        'l2_regularization': [0.1, 1.0]
    },
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)
hgb_grid.fit(train[enhanced_predictors], train["TargetBinary"])
print("Best HGB Parameters Found:", hgb_grid.best_params_)
print(f"HGB Tuned Grid Accuracy: {hgb_grid.best_score_:.4f}")

Best HGB Parameters Found: {'l2_regularization': 1.0, 'learning_rate': 0.05, 'max_depth': 4, 'max_iter': 120}
HGB Tuned Grid Accuracy: 0.6313


```markdown
## 6. Evaluation of Final Tuned Models on Chronological Test Set

We assess our optimized models against key performance indicators including **Balanced Accuracy** (to handle target imbalance) and **Precision (Win)** to avoid false-positive error types.
```

In [349]:
# Extracting final models
final_rf = rf_grid.best_estimator_
final_hgb = hgb_grid.best_estimator_

# Evaluate RandomForest
rf_preds = final_rf.predict(test[enhanced_predictors])
print("=== Tuned RandomForest Performance ===")
print(f"Accuracy Score         : {accuracy_score(test['TargetBinary'], rf_preds):.4f}")
print(f"Balanced Accuracy Score: {balanced_accuracy_score(test['TargetBinary'], rf_preds):.4f}")
print(f"Precision Score (Win)  : {precision_score(test['TargetBinary'], rf_preds, zero_division=0):.4f}")
print(f"Recall Score (Win)     : {recall_score(test['TargetBinary'], rf_preds, zero_division=0):.4f}")
print(f"F1 Score (Win)         : {f1_score(test['TargetBinary'], rf_preds, zero_division=0):.4f}")

=== Tuned RandomForest Performance ===
Accuracy Score         : 0.5898
Balanced Accuracy Score: 0.5715
Precision Score (Win)  : 0.5854
Recall Score (Win)     : 0.3556
F1 Score (Win)         : 0.4424


In [350]:
# Evaluate HistGradientBoosting
hgb_preds = final_hgb.predict(test[enhanced_predictors])
print("=== Tuned HistGradientBoosting Performance ===")
print(f"Accuracy Score         : {accuracy_score(test['TargetBinary'], hgb_preds):.4f}")
print(f"Balanced Accuracy Score: {balanced_accuracy_score(test['TargetBinary'], hgb_preds):.4f}")
print(f"Precision Score (Win)  : {precision_score(test['TargetBinary'], hgb_preds, zero_division=0):.4f}")
print(f"Recall Score (Win)     : {recall_score(test['TargetBinary'], hgb_preds, zero_division=0):.4f}")
print(f"F1 Score (Win)         : {f1_score(test['TargetBinary'], hgb_preds, zero_division=0):.4f}")

=== Tuned HistGradientBoosting Performance ===
Accuracy Score         : 0.5695
Balanced Accuracy Score: 0.5563
Precision Score (Win)  : 0.5400
Recall Score (Win)     : 0.4000
F1 Score (Win)         : 0.4596


```markdown
## 7. Live Prediction Simulator

Simulate future matchups by retrieving teams' latest computed rolling statistics, rest days, and Head-to-Head histories directly from our database.
```

In [356]:
def predict_match_live(home_team, away_team, weekday, kickoff_hour):
    # Re-train production classifiers on the complete enhanced feature dataset
    final_rf.fit(matches_enhanced[enhanced_predictors], matches_enhanced['TargetBinary'])
    final_hgb.fit(matches_enhanced[enhanced_predictors], matches_enhanced['TargetBinary'])

    try:
        home_code = matches_reset[matches_reset['HomeTeam'] == home_team]['HomeCode'].values[0]
        away_code = matches_reset[matches_reset['AwayTeam'] == away_team]['AwayCode'].values[0]
    except IndexError:
        print("Error: One of the specified teams was not found in the historical database. Please check spelling.")
        return

    home_latest = matches_enhanced[matches_enhanced['HomeTeam'] == home_team].sort_values(['Season', 'Month', 'Day']).tail(1)
    away_latest = matches_enhanced[matches_enhanced['AwayTeam'] == away_team].sort_values(['Season', 'Month', 'Day']).tail(1)

    if home_latest.empty or away_latest.empty:
        print("Error: Not enough historical rolling data to compute form for these teams.")
        return

    day_code = day_map.get(weekday, 5)
    custom_match = pd.DataFrame({
        'HomeCode': [home_code],
        'AwayCode': [away_code],
        'DayCode': [day_code],
        'Hour': [kickoff_hour],
        'Home_RestDays': [home_latest['Home_RestDays'].values[0]],
        'Away_RestDays': [away_latest['Away_RestDays'].values[0]],
        'H2H_Advantage': [home_latest['H2H_Advantage'].values[0]]
    })

    for col in enhanced_predictors:
        if col not in ['HomeCode', 'AwayCode', 'DayCode', 'Hour', 'Home_RestDays', 'Away_RestDays', 'H2H_Advantage']:
            custom_match[col] = home_latest[col].values[0] if 'Home_' in col else away_latest[col].values[0]

    rf_pred = final_rf.predict(custom_match[enhanced_predictors])[0]
    rf_prob = final_rf.predict_proba(custom_match[enhanced_predictors])[0]
    hgb_pred = final_hgb.predict(custom_match[enhanced_predictors])[0]
    hgb_prob = final_hgb.predict_proba(custom_match[enhanced_predictors])[0]

    outcome_map = {1: 'HOME WIN (1)', 0: 'DRAW OR AWAY WIN (0)'}

    print(f"=== Tuned RF Simulation: {home_team} vs {away_team} ===")
    print(f"Scheduled Day: {weekday} at {kickoff_hour:02d}:00")
    print(f"Model Prediction: {outcome_map[rf_pred]}")
    print(f"Confidence Probabilities -> Home Win: {rf_prob[1]*100:.2f}%, Draw/Away Win: {rf_prob[0]*100:.2f}%\n")

    print(f"=== Tuned HGB Simulation: {home_team} vs {away_team} ===")
    print(f"Scheduled Day: {weekday} at {kickoff_hour:02d}:00")
    print(f"Model Prediction: {outcome_map[hgb_pred]}")
    print(f"Confidence Probabilities -> Home Win: {hgb_prob[1]*100:.2f}%, Draw/Away Win: {hgb_prob[0]*100:.2f}%")

predict_match_live('Fulham', 'Newcastle', 'Sat', 17)

=== Tuned RF Simulation: Fulham vs Newcastle ===
Scheduled Day: Sat at 17:00
Model Prediction: DRAW OR AWAY WIN (0)
Confidence Probabilities -> Home Win: 49.71%, Draw/Away Win: 50.29%

=== Tuned HGB Simulation: Fulham vs Newcastle ===
Scheduled Day: Sat at 17:00
Model Prediction: DRAW OR AWAY WIN (0)
Confidence Probabilities -> Home Win: 49.15%, Draw/Away Win: 50.85%
